# V9 Residual-Shape Symbolic Regression Pilot: Iteration 1

## Purpose

V8 established that a hierarchical surrogate is viable, but its single global
shape expression underfits local high-stress structure. V9 keeps the completed
V8 case-mean, positive case-scale and global-shape formulas fixed, then searches
for one additional symbolic formula for the unexplained normalised residual:

`stress = case_mean + exp(case_log_scale) * (fixed_global_shape + residual_shape)`

This is a development-only pilot. It uses the frozen 119 training cases, all
elements in the 15 validation cases for formula selection, and the 15 internal
test cases once after selection. The 50 final-test cases remain sealed.


## What Changed From V8

- Four dimensionless boundary proxies are added to the reviewed 27 predictors.
  They express relative radial/axial position and proximity to the nearest
  observed case boundary. They are labelled as proxies because named FEM
  surfaces have not been confirmed.
- Every training case still contributes 5,000 discovery elements, but the
  quotas are 2,500 below P90, 500 from P90-P95, 1,000 from P95-P99 and 1,000
  above P99.
- The optimisation loss mass is fixed at 50%, 10%, 20% and 20% across those
  tiers. Sampling frequency and loss importance are therefore explicit and
  separately auditable.
- A Gaussian localisation operator `exp(-x^2)` is added. `tanh` is deliberately
  not added in the same experiment so any improvement remains attributable.
- Candidate selection uses a validation-only engineering score across RMSE,
  P95/P99 error, underprediction and hotspot metrics. Complexity is only a late
  tie-breaker; the old "within 3% RMSE choose simplest" rule is not used.
- Before PySR, nonlinear residual diagnostics compare geometry/boundary,
  physical-field and all-feature models. These diagnose learnability only and
  do not enter the final formula.


## 1. Fresh Kernel and Imports


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


def resolve_package_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "residual_shape_symbolic.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate the NotebookCT3 package root.")


PACKAGE_ROOT = resolve_package_root()
if str(PACKAGE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT / "src"))

from residual_shape_symbolic import (
    ResidualShapePilotConfig,
    output_directory,
    preflight_residual_pilot,
    run_residual_shape_pilot,
)

print("Package root:", PACKAGE_ROOT)
print("Python:", sys.executable)


Package root: /Users/novwin/Documents/University/University of Manchester/毕设项目/Nuclear graphite/NotebookCT3
Python: /usr/local/bin/python3.12


## 2. Locked Pilot Configuration

The pilot budget is `500 x 8 = 4,000` population iterations, split into five
recoverable 100-iteration segments. Each segment writes a PySR checkpoint and
can resume after a stalled worker or interrupted notebook. A formal 16,000-step
residual search is justified only if this pilot improves validation behaviour.


In [2]:
RUN_RESIDUAL_PILOT = True

CONFIG = ResidualShapePilotConfig(
    iteration=1,
    output_subdir="iteration_1",
    rows_per_training_case=5_000,
    total_niterations=500,
    populations=8,
    segment_niterations=100,
    population_size=40,
    ncycles_per_iteration=100,
    batch_size=50_000,
    maxsize=28,
    maxdepth=10,
    julia_threads=8,
    no_activity_timeout_seconds=45 * 60,
    segment_wall_timeout_seconds=3 * 60 * 60,
    watchdog_poll_seconds=60,
    max_attempts_per_segment=3,
    max_candidates_for_full_validation=16,
    hgb_max_iter=100,
    force_rebuild_training_cache=False,
)

OUTPUT_DIR = output_directory(PACKAGE_ROOT, CONFIG)
display(pd.DataFrame([CONFIG.__dict__]).T.rename(columns={0: "value"}))
print("Output directory:", OUTPUT_DIR)
print("Recoverable segments:", CONFIG.n_segments)
print("Planned population iterations:", CONFIG.target_population_iterations)


,value
iteration,1
output_subdir,iteration_1
rows_per_training_case,5000
total_niterations,500
populations,8
segment_niterations,100
population_size,40
ncycles_per_iteration,100
batch_size,50000
maxsize,28


Output directory: /Users/novwin/Documents/University/University of Manchester/毕设项目/Nuclear graphite/NotebookCT3/outputs/09_residual_shape_symbolic_pilot/iteration_1
Recoverable segments: 5
Planned population iterations: 4000


## 3. Preflight

This verifies the frozen 119/15/15 development split, 50 sealed final cases,
199 available FEM files, completed V8 formula artifacts, 31 V9 predictors and
the dedicated residual worker. It does not read final-test element data.


In [3]:
PREFLIGHT = preflight_residual_pilot(PACKAGE_ROOT, CONFIG)
display(PREFLIGHT["checks"])
print("Training cases:", len(PREFLIGHT["inputs"]["train_ids"]))
print("Validation cases:", len(PREFLIGHT["inputs"]["validation_ids"]))
print("Internal-test cases:", len(PREFLIGHT["inputs"]["internal_ids"]))
print("Sealed final-test cases:", len(PREFLIGHT["inputs"]["final_ids"]))


,check,value,expected,pass
0,training_cases,119,119,True
1,validation_cases,15,15,True
2,internal_test_cases,15,15,True
3,sealed_final_cases,50,50,True
4,residual_features,31,31,True
5,worker_script_exists,True,True,True
6,pilot_population_iterations,4000,4000,True
7,recoverable_segments,5,5,True


Training cases: 119
Validation cases: 15
Internal-test cases: 15
Sealed final-test cases: 50


## 4. Run or Resume the Complete Pilot

The controller first builds/reuses the fixed training cache, runs the residual
learnability diagnostic, executes five recoverable PySR segments, validates the
candidate frontier on complete validation cases, selects one formula using
validation data only, and finally reports it once on the internal-test cases.

Rerunning this notebook reuses the cache and every completed segment. Detailed
worker logs are stored under `segments/segment_XX/attempt_Y/`.


In [4]:
RESULT = None
if RUN_RESIDUAL_PILOT:
    RESULT = run_residual_shape_pilot(PACKAGE_ROOT, CONFIG)
    display(pd.DataFrame([RESULT]))
else:
    print("Pilot skipped because RUN_RESIDUAL_PILOT=False.")


[1/119] V9 residual sample: case_01
[2/119] V9 residual sample: case_10
[3/119] V9 residual sample: case_11
[4/119] V9 residual sample: case_12
[5/119] V9 residual sample: case_13
[6/119] V9 residual sample: case_14
[7/119] V9 residual sample: case_15
[8/119] V9 residual sample: case_16
[9/119] V9 residual sample: case_17
[10/119] V9 residual sample: case_19
[11/119] V9 residual sample: case_20
[12/119] V9 residual sample: case_21
[13/119] V9 residual sample: case_24
[14/119] V9 residual sample: case_27
[15/119] V9 residual sample: case_32
[16/119] V9 residual sample: case_34
[17/119] V9 residual sample: case_35
[18/119] V9 residual sample: case_36
[19/119] V9 residual sample: case_39
[20/119] V9 residual sample: case_40
[21/119] V9 residual sample: case_42
[22/119] V9 residual sample: case_43
[23/119] V9 residual sample: case_44
[24/119] V9 residual sample: case_47
[25/119] V9 residual sample: case_48
[26/119] V9 residual sample: case_49
[27/119] V9 residual sample: case_52
[28/119] V

/Users/novwin/Documents/University/University of Manchester/毕设项目/Nuclear graphite/NotebookCT3/src/residual_shape_symbolic.py:1135: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ True  True  True  True  True  True  True  True  True  True  True  True
  True]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_metrics.loc[valid.index, valid.columns] = valid
/Users/novwin/Documents/University/University of Manchester/毕设项目/Nuclear graphite/NotebookCT3/src/residual_shape_symbolic.py:1135: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[False False False False False False False False False False  True False
 False]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_metrics.loc[valid.index, valid.columns] = valid
/Users/novwin/Docu

[2/15] internal_test full-case V9 evaluation: case_28
[3/15] internal_test full-case V9 evaluation: case_57
[4/15] internal_test full-case V9 evaluation: case_66
[5/15] internal_test full-case V9 evaluation: case_71
[6/15] internal_test full-case V9 evaluation: case_82
[7/15] internal_test full-case V9 evaluation: case_93
[8/15] internal_test full-case V9 evaluation: case_98
[9/15] internal_test full-case V9 evaluation: case_102
[10/15] internal_test full-case V9 evaluation: case_117
[11/15] internal_test full-case V9 evaluation: case_147
[12/15] internal_test full-case V9 evaluation: case_159
[13/15] internal_test full-case V9 evaluation: case_196
[14/15] internal_test full-case V9 evaluation: case_197
[15/15] internal_test full-case V9 evaluation: case_199


,status,iteration,prototype_only,method,train_cases,validation_cases,internal_test_cases,final_test_cases_read,training_rows,planned_population_iterations,completed_segments,selected_residual_candidate,pilot_promotion_status,elapsed_seconds,output_directory
0,complete,1,True,fixed_v8_global_shape_plus_one_symbolic_residu...,119,15,15,0,595000,4000,5,2,prototype_selected_but_does_not_pass_all_pilot...,5682.423201,/Users/novwin/Documents/University/University ...


## 5. Saved Results and Decision Evidence


In [5]:
progress_path = OUTPUT_DIR / "search_progress.json"
if progress_path.exists():
    display(pd.DataFrame([json.loads(progress_path.read_text(encoding="utf-8"))]))

attempt_path = OUTPUT_DIR / "segment_attempt_audit.csv"
if attempt_path.exists():
    display(pd.read_csv(attempt_path).tail(20))

artifacts = {
    "completion": OUTPUT_DIR / "pilot_complete.json",
    "formula_text": OUTPUT_DIR / "selected_corrected_composite_formula.txt",
    "split_metrics": OUTPUT_DIR / "selected_formula_split_metrics.csv",
    "candidate_metrics": OUTPUT_DIR / "residual_candidate_validation_metrics.csv",
    "learnability": OUTPUT_DIR / "residual_learnability_split_metrics.csv",
    "sampling_audit": OUTPUT_DIR / "training_cache" / "training_sample_audit.csv",
}
display(pd.DataFrame([
    {"artifact": name, "exists": path.exists(), "path": str(path)}
    for name, path in artifacts.items()
]))

if artifacts["completion"].exists():
    print(artifacts["formula_text"].read_text(encoding="utf-8"))
    display(pd.read_csv(artifacts["learnability"]))
    display(pd.read_csv(artifacts["split_metrics"]))
    candidates = pd.read_csv(artifacts["candidate_metrics"])
    display(candidates.sort_values("engineering_selection_score").head(16))


,completed_segments,contiguous_completed_segments,total_segments,planned_population_iterations_completed,target_population_iterations,planned_progress_fraction,updated_at
0,"[1, 2, 3, 4, 5]",5,5,4000,4000,1.0,2026-08-18T09:58:51+0100


,segment,attempt,resume_from_checkpoint,return_code,termination_reason,stalled,wall_timed_out,elapsed_seconds,checkpoint_exists_after,worker_frontier_exists,finished_at
0,1,1,False,0,process_exit,False,False,1681.701784,True,True,2026-08-18T08:55:11+0100
1,2,1,True,0,process_exit,False,False,960.166605,True,True,2026-08-18T09:11:11+0100
2,3,1,True,0,process_exit,False,False,900.161436,True,True,2026-08-18T09:26:11+0100
3,4,1,True,0,process_exit,False,False,960.265813,True,True,2026-08-18T09:42:12+0100
4,5,1,True,0,process_exit,False,False,960.285306,True,True,2026-08-18T09:58:12+0100


,artifact,exists,path
0,completion,True,/Users/novwin/Documents/University/University ...
1,formula_text,True,/Users/novwin/Documents/University/University ...
2,split_metrics,True,/Users/novwin/Documents/University/University ...
3,candidate_metrics,True,/Users/novwin/Documents/University/University ...
4,learnability,True,/Users/novwin/Documents/University/University ...
5,sampling_audit,True,/Users/novwin/Documents/University/University ...


CT3 V9 residual-corrected symbolic stress formula

Case mean:
mu = -0.0116301024532845*temperature_mean + 0.0470427355318997*temperature_p95 - 3.16087946558646*weight_loss_rate_mean + 0.103449215368151*z_max - 0.870195660324862*Abs(4.60760803334225*fluence_rate_p95 - 20.3572633494299) - 100.195330145459

Positive case scale:
scale = exp(-1.47091131932291*rho_mean - 1011.416708227*theta_sin_std - 1.1690770556861*weight_loss_rate_mean + 0.904578545888617*z_mean + 9.23869712445276)

Fixed V8 global shape:
global_shape = 1.10317833861391*(0.55673575*(0.0509611749551904*rho - 9.32241465638866)*(0.0509611749551904*rho - 8.66867535638866) - 31.9731949759449*(theta_cos - 0.922666019258146)**2)*(Abs(0.0509611749551904*rho - 8.59575422638866) - 2.4201684) + 0.285739561816426

V9 residual shape:
residual_shape = -0.812977255834564*(z_within_case_z - 0.157641102303633)**2*((6.21020430729467*nearest_axial_boundary_fraction_proxy - 1.42454761725745)*exp(-0.659730203929873*(fluence_rate_within_case_z

,iteration,split,model,n_cases,n_elements_evaluated,micro_mae,micro_rmse,micro_r2,macro_mae,macro_rmse,...,mean_top5_actual_bias,mean_p95_relative_error,mean_p95_underprediction_fraction,mean_p99_relative_error,mean_p99_underprediction_fraction,mean_top5pct_hotspot_overlap,mean_top1pct_hotspot_overlap,mean_top1_recall_in_predicted_top5,max_prediction_abs_max_ratio,macro_rmse_improvement_vs_v8_fraction
0,1,validation,fixed_v8_global_trend,15,6005400,1.768228,2.403750,0.358755,1.768228,2.347327,...,-3.574538,0.128592,0.118436,0.214801,0.214801,0.307007,0.488994,0.600200,0.838627,0.000000
1,1,validation,residual_HGB_all_features,15,6005400,0.665477,0.918035,0.906467,0.665477,0.899609,...,0.269747,0.086205,0.000947,0.090279,0.001174,0.811353,0.869530,0.999334,1.301111,0.616752
2,1,validation,residual_HGB_geometry_boundary,15,6005400,0.722501,1.026712,0.883012,0.722501,1.000730,...,0.282861,0.103713,0.000000,0.102742,0.002236,0.760672,0.855911,0.998452,1.353035,0.573673
3,1,validation,residual_HGB_physical_fields,15,6005400,2.061633,2.718458,0.179855,2.061633,2.677946,...,-0.603319,0.281304,0.003312,0.064933,0.030196,0.424455,0.526424,0.790926,1.072027,-0.140849


,iteration,split,model,n_cases,n_elements_evaluated,micro_mae,micro_rmse,micro_r2,macro_mae,macro_rmse,...,mean_top5_actual_rmse,mean_top5_actual_bias,mean_p95_relative_error,mean_p95_underprediction_fraction,mean_p99_relative_error,mean_p99_underprediction_fraction,mean_top5pct_hotspot_overlap,mean_top1pct_hotspot_overlap,mean_top1_recall_in_predicted_top5,max_prediction_abs_max_ratio
0,1,internal_test,fixed_v8_global_trend,15,6005400,1.847777,2.471889,0.281895,1.847777,2.451950,...,5.110821,-3.972092,0.150652,0.150652,0.253503,0.253503,0.319945,0.447236,0.609590,1.166592
1,1,validation,fixed_v8_global_trend,15,6005400,1.768228,2.403750,0.358755,1.768228,2.347327,...,4.630008,-3.574538,0.128592,0.118436,0.214801,0.214801,0.307007,0.488994,0.600200,0.838627
2,1,internal_test,v9_trend_plus_symbolic_residual,15,6005400,1.840073,2.544181,0.239278,1.840073,2.482564,...,3.617868,-1.326105,0.146570,0.000000,0.044913,0.025605,0.467526,0.407925,0.789061,1.679942
3,1,validation,v9_trend_plus_symbolic_residual,15,6005400,1.793534,2.497747,0.307624,1.793534,2.443271,...,3.206142,-0.859080,0.199872,0.000000,0.069120,0.011107,0.470520,0.458675,0.834299,1.251237


,stage,run_id,candidate_index,complexity,loss,score,equation,formula_scaled_sympy,formula_original_variables,feature_support_json,...,gate_positive_macro_r2,gate_improves_v8_rmse_by_3pct,gate_p95_relative_error,gate_p99_relative_error,gate_p95_underprediction,gate_p99_underprediction,gate_top1_hotspot_overlap,gate_top1_recall,all_pilot_promotion_gates_pass,selected_candidate
10,residual_shape,v9_i1_residual_shape_pilot,2,17,0.700696,0.047224,z_within_case_z_scaled * (((nearest_axial_boun...,z_within_case_z_scaled*(nearest_axial_boundary...,-0.812977255834564*(z_within_case_z - 0.157641...,"[""fluence_rate_within_case_z_scaled"", ""nearest...",...,True,False,False,True,True,True,False,True,False,True
12,residual_shape,v9_i1_residual_shape_pilot,0,21,0.664493,0.011143,((z_within_case_z_scaled + 0.27480233) * ((((g...,(z_within_case_z_scaled + 0.27480233)*(z_withi...,-0.88007369535131*(0.933516437942946*z_within_...,"[""nearest_axial_boundary_fraction_proxy_scaled...",...,False,False,False,True,True,True,False,True,False,False
11,residual_shape,v9_i1_residual_shape_pilot,1,19,0.679468,0.015382,(((((nearest_axial_boundary_fraction_proxy_sca...,(z_within_case_z_scaled*(nearest_axial_boundar...,-0.811603501374446*(0.933516437942946*z_within...,"[""nearest_axial_boundary_fraction_proxy_scaled...",...,False,False,False,True,True,True,False,True,False,False
8,residual_shape,v9_i1_residual_shape_pilot,4,13,0.793582,0.007232,z_fraction_proxy_scaled * ((gauss(fluence_rate...,z_fraction_proxy_scaled*z_fraction_proxy_scale...,10.5240039922947*(-0.5707568 + exp(-0.65973020...,"[""fluence_rate_within_case_z_scaled"", ""z_fract...",...,False,False,False,True,True,True,False,True,False,False
5,residual_shape,v9_i1_residual_shape_pilot,7,10,0.837257,0.002473,(z_fraction_proxy_scaled * (abs(nearest_axial_...,z_fraction_proxy_scaled*z_fraction_proxy_scale...,10.5240039922947*(z_fraction_proxy - 0.5456495...,"[""nearest_axial_boundary_fraction_proxy_scaled...",...,False,False,False,True,True,True,False,False,False,False
4,residual_shape,v9_i1_residual_shape_pilot,8,9,0.839330,0.015919,(nearest_axial_boundary_fraction_proxy_scaled ...,(nearest_axial_boundary_fraction_proxy_scaled ...,-7.33766498475279*(z_fraction_proxy - 0.545649...,"[""nearest_axial_boundary_fraction_proxy_scaled...",...,False,False,False,True,True,True,False,True,False,False
6,residual_shape,v9_i1_residual_shape_pilot,6,11,0.821243,0.019312,gauss(fluence_rate_scaled) - gauss(nearest_axi...,-exp(-nearest_axial_boundary_fraction_proxy_sc...,0.453514790367153 - 1.02162831868781*exp(-38.5...,"[""fluence_rate_scaled"", ""nearest_axial_boundar...",...,False,False,False,True,True,True,False,False,False,False
9,residual_shape,v9_i1_residual_shape_pilot,3,15,0.770102,0.015017,(z_within_case_z_scaled * ((nearest_axial_boun...,z_within_case_z_scaled*(nearest_axial_boundary...,-0.658623313919049*(z_within_case_z - 0.157641...,"[""fluence_rate_scaled"", ""nearest_axial_boundar...",...,False,False,False,True,True,True,False,False,False,False
7,residual_shape,v9_i1_residual_shape_pilot,5,12,0.799342,0.027029,(gauss(fluence_rate_scaled) * abs(nearest_axia...,-1*0.5720946 + exp(-fluence_rate_scaled**2)*Ab...,-0.130953253961221 + 1.02162831868781*exp(-58....,"[""fluence_rate_scaled"", ""nearest_axial_boundar...",...,False,False,False,True,True,True,False,False,False,False
2,residual_shape,v9_i1_residual_shape_pilot,10,5,0.898365,0.023729,(nearest_radial_boundary_fraction_proxy_scaled...,(nearest_radial_boundary_fraction_proxy_scaled...,1.65355076540414*nearest_radial_boundary_fract...,"[""nearest_radial_boundary_fraction_proxy_scale...",...,False,False,False,True,True,False,False,False,False,False


## Interpretation Boundary

This notebook tests whether a readable residual correction can improve the V8
stress surrogate, particularly at P95/P99 and hotspot locations. Passing all
pilot gates supports a longer residual search. Failing them is still a valid
result: it means the present inputs/search space cannot justify more symbolic
search without another modelling change.

The output is not yet a lifetime model. Lifetime conversion still requires a
professor-confirmed strength, damage or failure relationship. The sealed
50-case final test must remain unused until formula structure and constants are
frozen across the full development workflow.
